# 03 — EDA: imaging embeddings (synthetic)

The simulator treats `imaging` as a **64-D embedding** with a weak sickling-related signal on early dimensions (see `synthetic.py`). Here we sanity-check that signal against severity.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koneke55/Mmvlm4SCD/blob/main/notebooks/03-eda-imaging.ipynb)

**Google Colab + GPU:** [Unsloth](https://unsloth.ai) documents a practical [Google Colab workflow](https://docs.unsloth.ai/get-started/install/google-colab) (free **T4** GPU tier, Runtime menu, run cells in order). Use it as the reference for attaching hardware acceleration.

**Note:** This repo does **not** depend on the `unsloth` pip package—only standard PyTorch + `pip install -e .`; the Unsloth guide covers Colab compute ergonomics.

**Local:** run `pip install -e .` from the repo root. **Colab:** run the environment cell below (clone under `/content` when needed).


## 1. Environment setup (Colab or local)

- **Colab:** optional `MMVLM_REPO_URL` for your fork; defaults to upstream.
- Installs this package editable (`pip install -e .`).


In [ ]:
import os
import subprocess
import sys


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _find_repo_root(start: str) -> str:
    cur = os.path.abspath(start)
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, "src", "mmvlm4scd")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        "Could not find mmvlm4scd package root (missing src/mmvlm4scd). "
        "Open the notebook from the repo or run the Colab clone cell."
    )


if _in_colab():
    REPO_URL = os.environ.get(
        "MMVLM_REPO_URL",
        "https://github.com/koneke55/Mmvlm4SCD.git",
    )
    DEST = "/content/Mmvlm4SCD"
    if not os.path.isdir(os.path.join(DEST, "src", "mmvlm4scd")):
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, DEST],
            stdout=subprocess.DEVNULL,
        )
    os.chdir(DEST)
    ROOT = DEST
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    ROOT = _find_repo_root(os.getcwd())
    os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

sys.path.insert(0, os.path.join(ROOT, "src"))
print("Repo root:", ROOT)


## 2. Accelerator check

Mirrors the GPU verification pattern recommended alongside [Unsloth's Colab instructions](https://docs.unsloth.ai/get-started/install/google-colab).


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("CPU-only runtime — for GPU follow Unsloth's Colab guide (Runtime → Change runtime type).")


## 3. Imports


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mmvlm4scd.data import generate_synthetic_cohort
from mmvlm4scd.data.synthetic import SCDSyntheticConfig


In [ ]:
cohort = generate_synthetic_cohort(SCDSyntheticConfig(n_patients=1500, seed=3))
img = cohort["imaging"]
sev = cohort["severity"]

plt.figure(figsize=(6, 4))
for k in range(3):
    m = sev == k
    plt.hist(img[m, 0], bins=30, alpha=0.45, density=True, label=f"severity {k}")
plt.xlabel("imaging[:, 0] (embedding dim)")
plt.ylabel("density")
plt.title("First embedding dimension vs severity")
plt.legend()
plt.tight_layout()
plt.show()

r = np.corrcoef(img[:, 0], sev)[0, 1]
print(f"Pearson corr(imaging[:,0], severity): {r:.3f}")


## Imaging dimension vs LDH (simulator couples dim 1 to labs)


In [ ]:
ldh = cohort["clinical"]["ldh_u_l"].values
plt.scatter(ldh, img[:, 1], c=sev, alpha=0.35, cmap="viridis")
plt.colorbar(label="severity")
plt.xlabel("LDH")
plt.ylabel("imaging[:, 1]")
plt.title("Embedding dim 1 vs LDH (colored by severity)")
plt.tight_layout()
plt.show()
